# XLSR per-segment evaluation — BLEU, chrF, TER, COMET

This notebook computes per-segment automatic evaluation scores for the XLSR pipeline outputs.

**Input:**
- `output/transcriptions_xlsr.csv` — produced by `xlsr_pipeline.ipynb`
- `data/reference_transcription_translation.csv` — versioned reference data

**Output:** `output/xlsr_eval_scores.xlsx` — one row per segment with all four metric scores appended.

**Metrics:**
- BLEU, chrF, TER: computed via `sacrebleu`. TER is an error metric (lower is better); BLEU and chrF are quality metrics (higher is better).
- COMET: computed via `unbabel-comet` using `Unbabel/wmt22-comet-da`. Requires source (reference French transcription), hypothesis (XLSR English translation), and reference (reference English translation).
- Sentence-level BLEU uses `effective_order=True`, which avoids returning zero for short segments where higher n-gram orders have no matches.
- `reference_transcription` is used as the COMET source field only; it is not used in BLEU, chrF, or TER computation.

**Runtime:** a GPU is recommended for COMET inference. A free Colab T4 is sufficient.

## Cell 1 — Install dependencies

Run once. After installation, **restart the runtime** before proceeding (Runtime > Restart session).
The restart is necessary because Colab pre-loads numpy at kernel startup; the version swap only takes effect in a fresh session.
After restarting, run all cells from Cell 3 onward — do not re-run this install cell.

In [ ]:
# Step 1: install unbabel-comet (its dependency pins are conservative)
!pip install unbabel-comet sacrebleu openpyxl xlsxwriter -q

# Step 2: restore numpy and protobuf to versions compatible with the rest of Colab
!pip install "numpy>=2.0" "protobuf>=5.0" -q

print("Installation complete. Restart the runtime before continuing.")

## Cell 2 — Configuration

Edit paths here if your files are not in the default repository locations.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
PIPELINE_OUTPUT = "output/transcriptions_xlsr.csv"                  # output of xlsr_pipeline.ipynb
REFERENCE_FILE  = "data/reference_transcription_translation.csv"    # versioned reference data
OUTPUT_FILE     = "output/xlsr_eval_scores.xlsx"

# Column names in the pipeline output
COL_SEGMENT   = "fichier"
COL_HYP_TRANS = "translation_en"
COL_HYP_TRANSCR = "transcription_fr"

# Column names in the reference file
COL_SRC = "reference_transcription"   # used as COMET source; not used in BLEU/chrF/TER
COL_REF = "reference_translation"     # used as metric reference for all four metrics

# COMET model
COMET_MODEL = "Unbabel/wmt22-comet-da"

print("Configuration loaded.")

## Cell 3 — Load and merge data

In [ ]:
import pandas as pd

df_pipeline  = pd.read_csv(PIPELINE_OUTPUT)
df_reference = pd.read_csv(REFERENCE_FILE)

print(f"Pipeline output:  {len(df_pipeline)} rows")
print(f"Reference file:   {len(df_reference)} rows")

# Merge on positional index (both files are in segment order)
if len(df_pipeline) != len(df_reference):
    raise ValueError(
        f"Row count mismatch: pipeline has {len(df_pipeline)} rows, "
        f"reference has {len(df_reference)} rows. Check both files."
    )

df = pd.concat([df_pipeline.reset_index(drop=True),
                df_reference.reset_index(drop=True)], axis=1)

print(f"Merged dataframe: {len(df)} rows")
print(f"Columns: {list(df.columns)}")

# Verify required columns
required = [COL_SRC, COL_REF, COL_HYP_TRANS]
missing  = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns after merge: {missing}")

# Normalise: cast to str, strip whitespace, replace 'nan' strings with empty string
def _norm(series):
    return series.astype(str).replace({"nan": ""}).str.strip()

for col in [COL_SRC, COL_REF, COL_HYP_TRANS, COL_HYP_TRANSCR]:
    if col in df.columns:
        df[col] = _norm(df[col])

mask_valid = df[COL_REF].ne("") & df[COL_HYP_TRANS].ne("")
n_invalid  = (~mask_valid).sum()
if n_invalid > 0:
    print(f"[WARNING] {n_invalid} row(s) have an empty reference or hypothesis — scores will be NaN for those rows.")
print(f"Valid rows for scoring: {mask_valid.sum()} / {len(df)}")

## Cell 4 — Per-segment sacrebleu scores (BLEU, chrF, TER)

In [ ]:
from sacrebleu.metrics import BLEU, CHRF, TER

# effective_order=True: use only n-gram orders present in the hypothesis.
# Prevents zero scores for short segments where 3- or 4-gram matches are absent.
bleu_metric = BLEU(effective_order=True, lowercase=True)
chrf_metric = CHRF()   # char_order=6, word_order=0, beta=2 (sacrebleu defaults)
ter_metric  = TER()    # lower = better (error rate)

def score_segment(hyp, ref):
    """Return (BLEU, chrF, TER) for one (hyp, ref) pair. Returns NaN if either is empty."""
    if not hyp or not ref:
        return float("nan"), float("nan"), float("nan")
    bleu = bleu_metric.sentence_score(hyp, [ref]).score
    chrf = chrf_metric.sentence_score(hyp, [ref]).score
    ter  = ter_metric.sentence_score(hyp, [ref]).score
    return round(bleu, 4), round(chrf, 4), round(ter, 4)

bleu_scores, chrf_scores, ter_scores = [], [], []
for hyp, ref in zip(df[COL_HYP_TRANS], df[COL_REF]):
    b, c, t = score_segment(hyp, ref)
    bleu_scores.append(b)
    chrf_scores.append(c)
    ter_scores.append(t)

df["bleu_xlsr"] = bleu_scores
df["chrf_xlsr"] = chrf_scores
df["ter_xlsr"]  = ter_scores

print("sacrebleu scores computed.")
print(df[[COL_HYP_TRANS, "bleu_xlsr", "chrf_xlsr", "ter_xlsr"]].head(5).to_string())

## Cell 5 — Load COMET model

In [ ]:
from comet import download_model, load_from_checkpoint

model_path  = download_model(COMET_MODEL)
comet_model = load_from_checkpoint(model_path)
print("COMET model loaded.")

## Cell 6 — Per-segment COMET scores


In [ ]:
comet_data    = []
comet_indices = []

for i, row in df.iterrows():
    src = row[COL_SRC]
    mt  = row[COL_HYP_TRANS]
    ref = row[COL_REF]
    if src and mt and ref:
        comet_data.append({"src": src, "mt": mt, "ref": ref})
        comet_indices.append(i)

print(f"Scoring {len(comet_data)} segments with COMET...")

output = comet_model.predict(comet_data, batch_size=8, gpus=1)

df["comet_xlsr"] = float("nan")
for idx, score in zip(comet_indices, output.scores):
    df.at[idx, "comet_xlsr"] = round(score, 4)

print("COMET scores computed.")
print(df[[COL_HYP_TRANS, "comet_xlsr"]].head(5).to_string())

## Cell 7 — Descriptive statistics

In [ ]:
score_cols = ["bleu_xlsr", "chrf_xlsr", "ter_xlsr", "comet_xlsr"]

print("XLSR per-segment score summary (50 segments)")
print("TER is an error metric: lower is better")
print(df[score_cols].describe().round(4).to_string())

## Cell 8 — Export to Excel

In [ ]:
import os
os.makedirs("output", exist_ok=True)

output_cols = []
for col in [COL_SEGMENT, COL_SRC, COL_HYP_TRANSCR, COL_REF, COL_HYP_TRANS]:
    if col in df.columns:
        output_cols.append(col)
output_cols += score_cols

df_out = df[output_cols].copy()

with pd.ExcelWriter(OUTPUT_FILE, engine="xlsxwriter") as writer:
    df_out.to_excel(writer, sheet_name="xlsr_scores", index=False)
    df_out[score_cols].describe().round(4).to_excel(writer, sheet_name="summary_stats")

print(f"Output saved to: {OUTPUT_FILE}")
print(f"  Sheet 'xlsr_scores'   : {len(df_out)} rows x {len(output_cols)} columns")
print(f"  Sheet 'summary_stats' : descriptive statistics for all four metrics")